In [1]:
cd ..

d:\VPBankHackathon


In [2]:
import time
import copy
import json
import re
from pymongo import MongoClient
from bson import ObjectId
from logger import _setup_logger
import config
from mongodb.mongo_pusher import MongoPusher
from utils import Utils
from blacklist_builder.article_extractor import ArticlePersonExtractor
from dynamodb.dynamo_pusher import DynamoPusher
from llm_model.bedrock_manager import BedrockModelManager
from dynamodb.base_dynamo import BaseDynamoDB
from llm_model.bedrock_manager import BedrockModelManager
from dynamodb.dynamo_table_checker import DynamoDBTableChecker
from dynamodb.base_dynamo import BaseDynamoDB
from dynamodb.media_service import MediaService
from dynamodb.dynamo_query import DynamoQuery
from dynamodb.table_personal2media import TablePersonal2Media
from dynamodb.table_personal_info import TablePersonalInfo
from dynamodb.table_org2media import TableOrg2Media
from dynamodb.table_org_info import TableOrganizationInfo
from dynamodb.table_adverse_media import TableAdverseMedia

In [3]:
logger = _setup_logger(__name__, config.LOG_LEVEL)

class ArticleRiskMatchingExtractor:
    def __init__(self, info_extractor, base_dynamo):
        """
        :param gemini_extractor: công cụ trích xuất từ Gemini API
        :param dynamo_pusher: đối tượng DynamoPusher (bắt buộc)
        """
        self.extractor = info_extractor
        self.base_dynamo = base_dynamo
        self.dynamo_pusher = DynamoPusher(dynamodb=self.base_dynamo)

        self.query = DynamoQuery(base_dynamo)

        self.personal_linker = TablePersonal2Media(self.query)
        self.personal_info = TablePersonalInfo(self.query)
        self.org_linker = TableOrg2Media(self.query)
        self.org_info = TableOrganizationInfo(self.query)
        self.media_service = MediaService(
            per_linker=self.personal_linker,
            per_info=self.personal_info,
            org_linker=self.org_linker,
            org_info=self.org_info
        )

    def extract_json_blocks(self, raw_text: str):
        json_blocks = json.loads(raw_text)
        if len(json_blocks) == 3:
            try:
                personal_info_json = json_blocks[0]
                org_info_json = json_blocks[1]
                risk_info_json = json_blocks[2]
            except json.JSONDecodeError as e:
                raise ValueError(f"JSON decode error: {e}")
        else:
            raise ValueError("Không tìm thấy đủ ba khối.")
        return personal_info_json, org_info_json, risk_info_json

    def create_adverse_media_item(self, risk_info: dict, partition_key: str, context: str): 
        risk_info_copy = copy.deepcopy(risk_info)

        unix_time = Utils.get_current_unix_time()
        item = {k: v for k, v in risk_info_copy.items() if k not in ["list_personal_risks", "list_organizer_risks"]}  
        item[partition_key] = partition_key + "_" + str(unix_time)
        item['content'] = context
        item["created_at"] = unix_time

        return item

    def create_personal_items(self, person_info: dict, partition_key: str):
        person_info_copy = copy.deepcopy(person_info)

        batch_unix_time = Utils.get_current_unix_time()
        per_id_gen_to_per_id = {}
        for item in person_info_copy:
            unix_time = Utils.get_current_unix_time()
            item[partition_key] = partition_key + "_" + str(unix_time)
            item["created_at"] = batch_unix_time
            
            per_id_gen_to_per_id[item["personal_id"]] = item[partition_key]
            item.pop("personal_id", None)

        return person_info_copy, per_id_gen_to_per_id

    def create_organization_items(self, org_info: dict, partition_key: str):
        org_info_copy = copy.deepcopy(org_info)

        batch_unix_time = Utils.get_current_unix_time()
        org_id_gen_to_per_id = {}
        for item in org_info_copy:
            unix_time = Utils.get_current_unix_time()
            item[partition_key] = partition_key + "_" + str(unix_time)
            item["created_at"] = batch_unix_time
            
            org_id_gen_to_per_id[item["organizer_id"]] = item[partition_key]
            item.pop("organizer_id", None)

        return org_info_copy, org_id_gen_to_per_id

    def create_personal2media_items(self, risk_info: dict, partition_key: str, per_id_gen_to_per_id: dict, media_id: str ):
        risk_info_copy = copy.deepcopy(risk_info)

        personal2media = risk_info_copy.get("list_personal_risks", [])
        personal2media_copy = copy.deepcopy(personal2media)

        batch_unix_time = Utils.get_current_unix_time()

        for item in personal2media_copy:
            item[partition_key] = partition_key + "_" + str(Utils.get_current_unix_time())
            item["media_id"] = media_id
            item["created_at"] = batch_unix_time
            item["per_id"] = per_id_gen_to_per_id.get(item["entity_id"], None)
            item.pop("entity_id", None)

        return personal2media_copy

    def create_org2media_items(self, risk_info: dict, partition_key: str, org_id_gen_to_per_id: dict, media_id: str ):
        risk_info_copy = copy.deepcopy(risk_info)

        organizer2media = risk_info_copy.get("list_organizer_risks", [])
        organizer2media_copy = copy.deepcopy(organizer2media)

        batch_unix_time = Utils.get_current_unix_time()
        for item in organizer2media_copy:
            item[partition_key] = partition_key + "_" + str(Utils.get_current_unix_time())
            item["media_id"] = media_id
            item["created_at"] = batch_unix_time
            item["org_id"] = org_id_gen_to_per_id.get(item["entity_id"], None)
            item.pop("entity_id", None)

        return organizer2media_copy

    def create_items_from_response(self, raw_text:str, article_text: str, partition_key: dict):
        personal_info_json, org_info_json, risk_info_json = self.extract_json_blocks(raw_text)
        adverse_media_item = self.create_adverse_media_item(risk_info_json, partition_key['media_config'], article_text)
        personal_info_items, per_id_gen_to_per_id = self.create_personal_items(personal_info_json, partition_key['person_config'])
        organization_info_items, org_id_gen_to_per_id = self.create_organization_items(org_info_json, partition_key['organization_config'])
        personal2media_items = self.create_personal2media_items(risk_info_json, partition_key['p2m_config'], per_id_gen_to_per_id, adverse_media_item['media_id'])
        org2media_items = self.create_org2media_items(risk_info_json, partition_key['o2m_config'], org_id_gen_to_per_id, adverse_media_item['media_id'])
        return {
            "adverse_media_item": adverse_media_item,
            "organization_info_items": organization_info_items,
            "personal_info_items": personal_info_items,
            "org2media_items": org2media_items,
            "personal2media_items": personal2media_items,
        }

    

    def extract_info_from_article(self, article_text: str, table_config: dict):
        logger.info("🧠 Gọi LLM để trích xuất bài báo...")
        answer, thinking = self.extractor.extract_from_article(article_text)

        logger.debug(f"[process_article] Raw response:\n{answer}")
        logger.info("📦 Đang xử lý kết quả trích xuất...")
        
        partition_key = {outer_key: list(inner_dict.values())[0] for outer_key, inner_dict in table_config.items()}
        logger.debug(f"[process_article] Partition keys: {partition_key}")

        items = self.create_items_from_response(answer, article_text, partition_key)
        logger.debug(f"[process_article] Adverse media: {items['adverse_media_item']}")
        logger.debug(f"[process_article] Personal info: {items['personal_info_items']}")
        logger.debug(f"[process_article] Organization info: {items['organization_info_items']}")
        logger.debug(f"[process_article] Personal to media: {items['personal2media_items']}")
        logger.debug(f"[process_article] Org to media: {items['org2media_items']}")

        return items
    
    def push_to_dynamodb(self, items: dict, table_config: dict):
        logger.info("📝 Đang lưu vào DynamoDB...")

        self.dynamo_pusher.insert(items['adverse_media_item'], table_config=table_config['media_config'])
        self.dynamo_pusher.insert(items['organization_info_items'], table_config=table_config['organization_config'])
        self.dynamo_pusher.insert(items['personal_info_items'], table_config=table_config['person_config'])
        self.dynamo_pusher.insert(items['org2media_items'], table_config=table_config['o2m_config'])
        self.dynamo_pusher.insert(items['personal2media_items'], table_config=table_config['p2m_config'])


        logger.info("✅ Xử lý bài báo hoàn tất.")
        

    def process_article(self, article_text: str,
                    table_config: dict):
        items = self.extract_info_from_article(article_text, table_config)
        self.push_to_dynamodb(items, table_config)

In [4]:
AWS_ACCESS_KEY = Utils.load_api_key_from_env("AWS_ACCESS_KEY")
AWS_SECRET_KEY = Utils.load_api_key_from_env("AWS_SECRET_KEY")
REGION = config.AWS_REGION

# Khởi tạo và insert
MODEL_REGION = config.AWS_VIRGINA_REGION
MODEL_ID = config.DEEPSEEK_MODEL_VIRGINA_ID

# Khởi tạo manager
bedrock_manager = BedrockModelManager(
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name=MODEL_REGION,
    default_model_id=MODEL_ID
)

extractor_prompt_file = config.PROMPT_EXTRACTOR_FILE
extractor_prompt = Utils.load_text(extractor_prompt_file)
logger.info(f"Đã tải prompt từ {extractor_prompt_file}")

extractor = ArticlePersonExtractor(model_manager=bedrock_manager, prompt_template=extractor_prompt)
logger.info("Đã khởi tạo ArticlePersonExtractor")

base_dynamo = BaseDynamoDB(
    region_name=REGION,
    access_key=AWS_ACCESS_KEY,
    secret_key=AWS_SECRET_KEY
)

processor = ArticleRiskMatchingExtractor(
    info_extractor=extractor,
    base_dynamo=base_dynamo.dynamodb
)

context_file = 'data/contents_old.json'
bucket_name = "team253"
key = "adverse_media_data/case1.json"

contents = Utils.fetch_json_from_s3(bucket_name, key)
logger.info(f"Đã load {len(contents)} bài báo từ file {context_file}")

content = contents[0]

table_config = {
    'media_config': {"adverse_media": "media_id"},
    'person_config': {"personal_info": "per_id"},
    'organization_config': {"organization_info": "org_id"},
    'p2m_config':{"personal2media": "p2m_id"},
    'o2m_config': {"org2media": "o2m_id"}
}
table_media_config={"adverse_media": "media_id"}
table_person_config={"personal_info": "per_id"}
table_organization_config={"organization_info": "org_id"}
table_p2m_config={"personal2media": "p2m_id"}
table_o2m_config={"org2media": "o2m_id"}

<DEBUG-utils> - [load_api_key_from_env] ✅ Đã load key 'AWS_ACCESS_KEY' từ môi trường
<DEBUG-utils> - [load_api_key_from_env] ✅ Đã load key 'AWS_SECRET_KEY' từ môi trường
<DEBUG-utils> - [load_text] ✅ Đã load file văn bản: prompts/prompt_extractor.txt
<INFO-__main__> - Đã tải prompt từ prompts/prompt_extractor.txt
<INFO-__main__> - Đã khởi tạo ArticlePersonExtractor
<INFO-dynamodb.base_dynamo> - Connected to DynamoDB in region ap-southeast-1
<INFO-__main__> - Đã load 4 bài báo từ file data/contents_old.json


<!-- # Call Extractor -->

In [5]:
# article_text = contents[1]

In [6]:
# logger.info("🧠 Gọi LLM để trích xuất bài báo...")
# answer, thinking = extractor.extract_from_article(article_text)

# logger.debug(f"[process_article] Raw response:\n{answer}")
# logger.info("📦 Đang xử lý kết quả trích xuất...")

# Old Answer

In [7]:
old_answer = '[\n  [\n    {\n      "personal_id": "P001",\n      "full_name": "Trương Mỹ Lan",\n      "birth_year_or_age": null,\n      "gender": "Female",\n      "occupation_or_position": "Chủ tịch hội đồng quản trị",\n      "organization": "Tập đoàn Vạn Thịnh Phát",\n      "hometown_or_residence": null,\n      "personal_relationships": "Bị cáo chính; Cổ đông lớn Ngân hàng SCB; Chỉ đạo các bị cáo Đinh Văn Thành, Bùi Anh Dũng, Võ Tấn Hoàng Văn, Tạ Chiêu Trung, Trương Khánh Hoàng, Trần Thị Mỹ Dung"\n    },\n    {\n      "personal_id": "P002",\n      "full_name": "Đinh Văn Thành",\n      "birth_year_or_age": null,\n      "gender": "Male",\n      "occupation_or_position": null,\n      "organization": null,\n      "hometown_or_residence": null,\n      "personal_relationships": "Đồng phạm của Trương Mỹ Lan"\n    },\n    {\n      "personal_id": "P003",\n      "full_name": "Bùi Anh Dũng",\n      "birth_year_or_age": null,\n      "gender": "Male",\n      "occupation_or_position": null,\n      "organization": null,\n      "hometown_or_residence": null,\n      "personal_relationships": "Đồng phạm của Trương Mỹ Lan"\n    },\n    {\n      "personal_id": "P004",\n      "full_name": "Võ Tấn Hoàng Văn",\n      "birth_year_or_age": null,\n      "gender": "Male",\n      "occupation_or_position": null,\n      "organization": null,\n      "hometown_or_residence": null,\n      "personal_relationships": "Đồng phạm của Trương Mỹ Lan"\n    },\n    {\n      "personal_id": "P005",\n      "full_name": "Tạ Chiêu Trung",\n      "birth_year_or_age": null,\n      "gender": "Male",\n      "occupation_or_position": null,\n      "organization": null,\n      "hometown_or_residence": null,\n      "personal_relationships": "Đồng phạm của Trương Mỹ Lan"\n    },\n    {\n      "personal_id": "P006",\n      "full_name": "Trương Khánh Hoàng",\n      "birth_year_or_age": null,\n      "gender": "Male",\n      "occupation_or_position": null,\n      "organization": null,\n      "hometown_or_residence": null,\n      "personal_relationships": "Đồng phạm của Trương Mỹ Lan"\n    },\n    {\n      "personal_id": "P007",\n      "full_name": "Trần Thị Mỹ Dung",\n      "birth_year_or_age": null,\n      "gender": "Female",\n      "occupation_or_position": null,\n      "organization": null,\n      "hometown_or_residence": null,\n      "personal_relationships": "Đồng phạm của Trương Mỹ Lan"\n    },\n    {\n      "personal_id": "P008",\n      "full_name": "Huỳnh Thanh Duyên",\n      "birth_year_or_age": null,\n      "gender": "Female",\n      "occupation_or_position": "Thẩm phán chủ tọa",\n      "organization": "Tòa án nhân dân cấp cao tại TP.HCM",\n      "hometown_or_residence": null,\n      "personal_relationships": "Chủ tọa phiên tòa phúc thẩm"\n    },\n    {\n      "personal_id": "P009",\n      "full_name": "Phạm Công Mười",\n      "birth_year_or_age": null,\n      "gender": "Male",\n      "occupation_or_position": "Thẩm phán",\n      "organization": "Tòa án nhân dân cấp cao tại TP.HCM",\n      "hometown_or_residence": null,\n      "personal_relationships": "Thành viên hội đồng xét xử"\n    },\n    {\n      "personal_id": "P010",\n      "full_name": "Lê Thành Long",\n      "birth_year_or_age": null,\n      "gender": "Male",\n      "occupation_or_position": "Thẩm phán",\n      "organization": "Tòa án nhân dân cấp cao tại TP.HCM",\n      "hometown_or_residence": null,\n      "personal_relationships": "Thành viên hội đồng xét xử"\n    },\n    {\n      "personal_id": "P011",\n      "full_name": "Võ Phong Lưu",\n      "birth_year_or_age": null,\n      "gender": "Male",\n      "occupation_or_position": "Kiểm sát viên",\n      "organization": "Viện Kiểm sát nhân dân cấp cao tại TP.HCM",\n      "hometown_or_residence": null,\n      "personal_relationships": "Tham gia phiên tòa"\n    },\n    {\n      "personal_id": "P012",\n      "full_name": "Đặng Quốc Việt",\n      "birth_year_or_age": null,\n      "gender": "Male",\n      "occupation_or_position": "Kiểm sát viên",\n      "organization": "Viện Kiểm sát nhân dân cấp cao tại TP.HCM",\n      "hometown_or_residence": null,\n      "personal_relationships": "Tham gia phiên tòa"\n    },\n    {\n      "personal_id": "P013",\n      "full_name": "Đỗ Phước Trung",\n      "birth_year_or_age": null,\n      "gender": "Male",\n      "occupation_or_position": "Kiểm sát viên",\n      "organization": "Viện Kiểm sát nhân dân cấp cao tại TP.HCM",\n      "hometown_or_residence": null,\n      "personal_relationships": "Tham gia phiên tòa"\n    },\n    {\n      "personal_id": "P014",\n      "full_name": "Nguyễn Sơn Hoa",\n      "birth_year_or_age": null,\n      "gender": "Male",\n      "occupation_or_position": null,\n      "organization": "Công ty CP Quốc Cường Gia Lai",\n      "hometown_or_residence": null,\n      "personal_relationships": "Người có quyền lợi, nghĩa vụ liên quan"\n    }\n  ],\n  [\n    {\n      "organizer_id": "O001",\n      "full_name": "Tập đoàn Vạn Thịnh Phát",\n      "birth_year_or_age": null,\n      "gender": null,\n      "occupation_or_position": null,\n      "organization": null,\n      "hometown_or_residence": null,\n      "personal_relationships": "Có Trương Mỹ Lan là chủ tịch"\n    },\n    {\n      "organizer_id": "O002",\n      "full_name": "Ngân hàng SCB",\n      "birth_year_or_age": null,\n      "gender": null,\n      "occupation_or_position": null,\n      "organization": null,\n      "hometown_or_residence": null,\n      "personal_relationships": "Bị chiếm đoạt tiền bởi Trương Mỹ Lan và đồng phạm"\n    },\n    {\n      "organizer_id": "O003",\n      "full_name": "Công ty CP Quốc Cường Gia Lai",\n      "birth_year_or_age": null,\n      "gender": null,\n      "occupation_or_position": null,\n      "organization": null,\n      "hometown_or_residence": null,\n      "personal_relationships": "Bên có quyền lợi, nghĩa vụ liên quan"\n    },\n    {\n      "organizer_id": "O004",\n      "full_name": "Công ty CP T&H Hạ Long",\n      "birth_year_or_age": null,\n      "gender": null,\n      "occupation_or_position": null,\n      "organization": null,\n      "hometown_or_residence": null,\n      "personal_relationships": "Bên có quyền lợi, nghĩa vụ liên quan"\n    },\n    {\n      "organizer_id": "O005",\n      "full_name": "Công ty Âu Lạc Quảng Ninh",\n      "birth_year_or_age": null,\n      "gender": null,\n      "occupation_or_position": null,\n      "organization": null,\n      "hometown_or_residence": null,\n      "personal_relationships": "Bên có quyền lợi, nghĩa vụ liên quan"\n    }\n  ],\n  {\n    "news_sentiment_type": "Negative",\n    "recency": "recent",\n    "source_credibility": "High",\n    "list_personal_risks": [\n      {\n        "entity_id": "P001",\n        "entity_name": "Trương Mỹ Lan",\n        "is_individual": true,\n        "role_in_event": "Bị cáo chính",\n        "violation_type": "Tham ô tài sản, Hối lộ, Vi phạm quy định cho vay",\n        "customer_role": "Chủ mưu",\n        "legal_status": "Đã kết án",\n        "source_level": "Other"\n      },\n      {\n        "entity_id": "P002",\n        "entity_name": "Đinh Văn Thành",\n        "is_individual": true,\n        "role_in_event": "Đồng phạm",\n        "violation_type": "Tham ô tài sản",\n        "customer_role": "Tham gia",\n        "legal_status": "Đã kết án",\n        "source_level": "Other"\n      },\n      {\n        "entity_id": "P003",\n        "entity_name": "Bùi Anh Dũng",\n        "is_individual": true,\n        "role_in_event": "Đồng phạm",\n        "violation_type": "Tham ô tài sản",\n        "customer_role": "Tham gia",\n        "legal_status": "Đã kết án",\n        "source_level": "Other"\n      },\n      {\n        "entity_id": "P004",\n        "entity_name": "Võ Tấn Hoàng Văn",\n        "is_individual": true,\n        "role_in_event": "Đồng phạm",\n        "violation_type": "Tham ô tài sản",\n        "customer_role": "Tham gia",\n        "legal_status": "Đã kết án",\n        "source_level": "Other"\n      },\n      {\n        "entity_id": "P005",\n        "entity_name": "Tạ Chiêu Trung",\n        "is_individual": true,\n        "role_in_event": "Đồng phạm",\n        "violation_type": "Tham ô tài sản",\n        "customer_role": "Tham gia",\n        "legal_status": "Đã kết án",\n        "source_level": "Other"\n      },\n      {\n        "entity_id": "P006",\n        "entity_name": "Trương Khánh Hoàng",\n        "is_individual": true,\n        "role_in_event": "Đồng phạm",\n        "violation_type": "Tham ô tài sản",\n        "customer_role": "Tham gia",\n        "legal_status": "Đã kết án",\n        "source_level": "Other"\n      },\n      {\n        "entity_id": "P007",\n        "entity_name": "Trần Thị Mỹ Dung",\n        "is_individual": true,\n        "role_in_event": "Đồng phạm",\n        "violation_type": "Tham ô tài sản",\n        "customer_role": "Tham gia",\n        "legal_status": "Đã kết án",\n        "source_level": "Other"\n      },\n      {\n        "entity_id": "P008",\n        "entity_name": "Huỳnh Thanh Duyên",\n        "is_individual": true,\n        "role_in_event": "Thẩm phán chủ tọa",\n        "violation_type": "Không có hành vi phạm được đề cập",\n        "customer_role": "Không có liên quan rõ ràng",\n        "legal_status": "Tin chưa rõ ràng",\n        "source_level": "Other"\n      },\n      {\n        "entity_id": "P009",\n        "entity_name": "Phạm Công Mười",\n        "is_individual": true,\n        "role_in_event": "Thẩm phán",\n        "violation_type": "Không có hành vi phạm được đề cập",\n        "customer_role": "Không có liên quan rõ ràng",\n        "legal_status": "Tin chưa rõ ràng",\n        "source_level": "Other"\n      },\n      {\n        "entity_id": "P010",\n        "entity_name": "Lê Thành Long",\n        "is_individual": true,\n        "role_in_event": "Thẩm phán",\n        "violation_type": "Không có hành vi phạm được đề cập",\n        "customer_role": "Không có liên quan rõ ràng",\n        "legal_status": "Tin chưa rõ ràng",\n        "source_level": "Other"\n      },\n      {\n        "entity_id": "P011",\n        "entity_name": "Võ Phong Lưu",\n        "is_individual": true,\n        "role_in_event": "Kiểm sát viên",\n        "violation_type": "Không có hành vi phạm được đề cập",\n        "customer_role": "Không có liên quan rõ ràng",\n        "legal_status": "Tin chưa rõ ràng",\n        "source_level": "Other"\n      },\n      {\n        "entity_id": "P012",\n        "entity_name": "Đặng Quốc Việt",\n        "is_individual": true,\n        "role_in_event": "Kiểm sát viên",\n        "violation_type": "Không có hành vi phạm được đề cập",\n        "customer_role": "Không có liên quan rõ ràng",\n        "legal_status": "Tin chưa rõ ràng",\n        "source_level": "Other"\n      },\n      {\n        "entity_id": "P013",\n        "entity_name": "Đỗ Phước Trung",\n        "is_individual": true,\n        "role_in_event": "Kiểm sát viên",\n        "violation_type": "Không có hành vi phạm được đề cập",\n        "customer_role": "Không có liên quan rõ ràng",\n        "legal_status": "Tin chưa rõ ràng",\n        "source_level": "Other"\n      },\n      {\n        "entity_id": "P014",\n        "entity_name": "Nguyễn Sơn Hoa",\n        "is_individual": true,\n        "role_in_event": "Người có quyền lợi liên quan",\n        "violation_type": "Không có hành vi phạm được đề cập",\n        "customer_role": "Bên liên quan bị động",\n        "legal_status": "Tin chưa rõ ràng",\n        "source_level": "Other"\n      }\n    ],\n    "list_organizer_risks": [\n      {\n        "entity_id": "O001",\n        "entity_name": "Tập đoàn Vạn Thịnh Phát",\n        "is_individual": false,\n        "role_in_event": "Liên quan đến vụ án",\n        "violation_type": "Tham ô tài sản",\n        "customer_role": "Tổ chức thực hiện hành vi vi phạm",\n        "legal_status": "Đang trong quá trình điều tra",\n        "source_level": "Other"\n      },\n      {\n        "entity_id": "O002",\n        "entity_name": "Ngân hàng SCB",\n        "is_individual": false,\n        "role_in_event": "Bị hại",\n        "violation_type": "Tham ô tài sản",\n        "customer_role": "Bên liên quan bị động",\n        "legal_status": "Đang trong quá trình điều tra",\n        "source_level": "Other"\n      },\n      {\n        "entity_id": "O003",\n        "entity_name": "Công ty CP Quốc Cường Gia Lai",\n        "is_individual": false,\n        "role_in_event": "Bên liên quan",\n        "violation_type": "Không có hành vi phạm được đề cập",\n        "customer_role": "Bên liên quan bị động",\n        "legal_status": "Tin chưa rõ ràng",\n        "source_level": "Other"\n      },\n      {\n        "entity_id": "O004",\n        "entity_name": "Công ty CP T&H Hạ Long",\n        "is_individual": false,\n        "role_in_event": "Bên liên quan",\n        "violation_type": "Không có hành vi phạm được đề cập",\n        "customer_role": "Bên liên quan bị động",\n        "legal_status": "Tin chưa rõ ràng",\n        "source_level": "Other"\n      },\n      {\n        "entity_id": "O005",\n        "entity_name": "Công ty Âu Lạc Quảng Ninh",\n        "is_individual": false,\n        "role_in_event": "Bên liên quan",\n        "violation_type": "Không có hành vi phạm được đề cập",\n        "customer_role": "Bên liên quan bị động",\n        "legal_status": "Tin chưa rõ ràng",\n        "source_level": "Other"\n      }\n    ]\n  }\n]'

# Extract json

In [8]:
new_personal_info_json, new_org_info_json, new_risk_info_json = processor.extract_json_blocks(old_answer)

In [9]:
new_personal_info_json

[{'personal_id': 'P001',
  'full_name': 'Trương Mỹ Lan',
  'birth_year_or_age': None,
  'gender': 'Female',
  'occupation_or_position': 'Chủ tịch hội đồng quản trị',
  'organization': 'Tập đoàn Vạn Thịnh Phát',
  'hometown_or_residence': None,
  'personal_relationships': 'Bị cáo chính; Cổ đông lớn Ngân hàng SCB; Chỉ đạo các bị cáo Đinh Văn Thành, Bùi Anh Dũng, Võ Tấn Hoàng Văn, Tạ Chiêu Trung, Trương Khánh Hoàng, Trần Thị Mỹ Dung'},
 {'personal_id': 'P002',
  'full_name': 'Đinh Văn Thành',
  'birth_year_or_age': None,
  'gender': 'Male',
  'occupation_or_position': None,
  'organization': None,
  'hometown_or_residence': None,
  'personal_relationships': 'Đồng phạm của Trương Mỹ Lan'},
 {'personal_id': 'P003',
  'full_name': 'Bùi Anh Dũng',
  'birth_year_or_age': None,
  'gender': 'Male',
  'occupation_or_position': None,
  'organization': None,
  'hometown_or_residence': None,
  'personal_relationships': 'Đồng phạm của Trương Mỹ Lan'},
 {'personal_id': 'P004',
  'full_name': 'Võ Tấn H

In [10]:
new_org_info_json

[{'organizer_id': 'O001',
  'full_name': 'Tập đoàn Vạn Thịnh Phát',
  'birth_year_or_age': None,
  'gender': None,
  'occupation_or_position': None,
  'organization': None,
  'hometown_or_residence': None,
  'personal_relationships': 'Có Trương Mỹ Lan là chủ tịch'},
 {'organizer_id': 'O002',
  'full_name': 'Ngân hàng SCB',
  'birth_year_or_age': None,
  'gender': None,
  'occupation_or_position': None,
  'organization': None,
  'hometown_or_residence': None,
  'personal_relationships': 'Bị chiếm đoạt tiền bởi Trương Mỹ Lan và đồng phạm'},
 {'organizer_id': 'O003',
  'full_name': 'Công ty CP Quốc Cường Gia Lai',
  'birth_year_or_age': None,
  'gender': None,
  'occupation_or_position': None,
  'organization': None,
  'hometown_or_residence': None,
  'personal_relationships': 'Bên có quyền lợi, nghĩa vụ liên quan'},
 {'organizer_id': 'O004',
  'full_name': 'Công ty CP T&H Hạ Long',
  'birth_year_or_age': None,
  'gender': None,
  'occupation_or_position': None,
  'organization': None,
  

# Check for old medias

## Prepare old list media_id to compare

In [11]:
query = DynamoQuery(base_dynamo.dynamodb)
table_adverse_media = TableAdverseMedia(query)

media_ids = table_adverse_media.get_all_media_ids()
for mid in media_ids:
    print(mid)

media_id_1752384078640333
media_id_1752384042694009
media_id_1752384018355620
media_id_1752383993651068


## Compare one old media

### Prepare new data to prompt compare

In [12]:
article_text = contents[1]

In [13]:
new_article_text = article_text

In [14]:
new_personal_info = new_personal_info_json

In [40]:
print(Utils.json_to_str(new_org_info))

[
  {
    "organizer_id": "O001",
    "full_name": "Tập đoàn Vạn Thịnh Phát",
    "birth_year_or_age": null,
    "gender": null,
    "occupation_or_position": null,
    "organization": null,
    "hometown_or_residence": null,
    "personal_relationships": "Có Trương Mỹ Lan là chủ tịch"
  },
  {
    "organizer_id": "O002",
    "full_name": "Ngân hàng SCB",
    "birth_year_or_age": null,
    "gender": null,
    "occupation_or_position": null,
    "organization": null,
    "hometown_or_residence": null,
    "personal_relationships": "Bị chiếm đoạt tiền bởi Trương Mỹ Lan và đồng phạm"
  },
  {
    "organizer_id": "O003",
    "full_name": "Công ty CP Quốc Cường Gia Lai",
    "birth_year_or_age": null,
    "gender": null,
    "occupation_or_position": null,
    "organization": null,
    "hometown_or_residence": null,
    "personal_relationships": "Bên có quyền lợi, nghĩa vụ liên quan"
  },
  {
    "organizer_id": "O004",
    "full_name": "Công ty CP T&H Hạ Long",
    "birth_year_or_age": nul

In [39]:
new_org_info = new_org_info_json

### Prepare old data one media from media_id

In [16]:
media_id = media_ids[0]
print(media_id)

media_id_1752384078640333


In [17]:
query = DynamoQuery(base_dynamo.dynamodb)
personal_linker = TablePersonal2Media(query)
personal_info = TablePersonalInfo(query)
org_linker = TableOrg2Media(query)
org_info = TableOrganizationInfo(query)
media_service = MediaService(
    per_linker=personal_linker,
    per_info=personal_info,
    org_linker=org_linker,
    org_info=org_info
)        

In [18]:
for attr in dir(TableAdverseMedia):
    if callable(getattr(TableAdverseMedia, attr)) and not attr.startswith("__"):
        print(f"Method: {attr}")

Method: get_all_media_ids
Method: get_document_by_media_id


In [19]:
old_article_text = table_adverse_media.get_document_by_media_id(media_id)['content']
print(old_article_text)


Phó chủ nhiệm Ủy ban Pháp luật và Tư pháp của Quốc hội Nguyễn Phương Thủy giải đáp câu hỏi bà Trương Mỹ Lan có được chuyển từ án tử hình xuống chung thân. Trưa 27-6, tổng thư ký Quốc hội tổ chức họp báo công bố kết quả kỳ họp thứ 9, Quốc hội khóa XV. Tại cuộc họp báo, một phóng viên nước ngoài đặt vấn đề Luật sửa đổi, bổ sung một số điều của Bộ luật Hình sự vừa được Quốc hội thông qua tại kỳ họp đã bỏ án tử hình với 8 tội danh. Trong đó về điều khoản chuyển tiếp, luật quy định hình phạt tử hình đã tuyên trước ngày 1-7-2025 đối với người phạm 8 tội danh nêu trên của Bộ luật Hình sự mà chưa thi hành án thì không thi hành và chánh án TAND tối cao quyết định chuyển hình phạt tử hình thành tù chung thân. Phóng viên hỏi trường hợp bà Trương Mỹ Lan (người bị tuyên án tử hình trong vụ án Vạn Thịnh Phát - giai đoạn 1) có được áp dụng quy định chuyển xuống hình phạt chung thân hay không? Trả lời vấn đề này, bà Nguyễn Phương Thủy - phó chủ nhiệm Ủy ban Pháp luật và Tư pháp của Quốc hội - cho biế

In [20]:
old_personal_info = media_service.get_personal_info_by_media(media_id)
print(json.dumps(old_personal_info, ensure_ascii=False, indent=2, default=str))

[
  {
    "organization": "Ủy ban Pháp luật và Tư pháp của Quốc hội",
    "created_at": "1752384078640383",
    "per_id": "per_id_1752384078640384",
    "full_name": "Nguyễn Phương Thủy",
    "occupation_or_position": "Phó chủ nhiệm Ủy ban Pháp luật và Tư pháp của Quốc hội",
    "hometown_or_residence": null,
    "gender": "Female",
    "personal_relationships": null,
    "birth_year_or_age": null
  },
  {
    "organization": "Tập đoàn Vạn Thịnh Phát",
    "created_at": "1752384078640383",
    "per_id": "per_id_1752384078640389",
    "full_name": "Trương Mỹ Lan",
    "occupation_or_position": "Chủ tịch Tập đoàn Vạn Thịnh Phát",
    "hometown_or_residence": null,
    "gender": "Female",
    "personal_relationships": "Bị cáo trong vụ án Vạn Thịnh Phát",
    "birth_year_or_age": null
  }
]


In [21]:
old_org_info = media_service.get_org_info_by_media(media_id)
print(json.dumps(old_org_info, ensure_ascii=False, indent=2, default=str))

[
  {
    "org_id": "org_id_1752384078640468",
    "organization": null,
    "created_at": "1752384078640466",
    "full_name": "Ủy ban Pháp luật và Tư pháp của Quốc hội",
    "occupation_or_position": "Cơ quan quản lý pháp luật",
    "hometown_or_residence": null,
    "gender": null,
    "personal_relationships": null,
    "birth_year_or_age": null
  },
  {
    "org_id": "org_id_1752384078640473",
    "organization": null,
    "created_at": "1752384078640466",
    "full_name": "Tòa án nhân dân cấp cao tại TP.HCM",
    "occupation_or_position": "Cơ quan xét xử",
    "hometown_or_residence": null,
    "gender": null,
    "personal_relationships": null,
    "birth_year_or_age": null
  },
  {
    "org_id": "org_id_1752384078640476",
    "organization": null,
    "created_at": "1752384078640466",
    "full_name": "TAND tối cao",
    "occupation_or_position": "Cơ quan tư pháp tối cao",
    "hometown_or_residence": null,
    "gender": null,
    "personal_relationships": null,
    "birth_year

## Call Prompt bedrock compare

In [22]:
AWS_ACCESS_KEY = Utils.load_api_key_from_env("AWS_ACCESS_KEY")
AWS_SECRET_KEY = Utils.load_api_key_from_env("AWS_SECRET_KEY")
REGION = config.AWS_VIRGINA_REGION
MODEL_ID = config.DEEPSEEK_MODEL_VIRGINA_ID

# Khởi tạo manager
manager = BedrockModelManager(
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name=REGION,
    default_model_id=MODEL_ID
)

<DEBUG-utils> - [load_api_key_from_env] ✅ Đã load key 'AWS_ACCESS_KEY' từ môi trường
<DEBUG-utils> - [load_api_key_from_env] ✅ Đã load key 'AWS_SECRET_KEY' từ môi trường


In [23]:
prompt = f"""
Mục tiêu:

Sử dụng cả hai thông tin bài báo và thông tin khách hàng để so sánh danh sách personal_info từ dữ liệu cũ và dữ liệu mới nhằm xác định các cá nhân trùng lặp. Đối với mỗi cá nhân trùng lặp được tìm thấy, hãy trả về một đối tượng JSON chứa personal_id của họ từ dữ liệu mới, per_unix_id của họ từ dữ liệu cũ, và một bản tổng hợp thông tin cá nhân của họ từ cả hai nguồn (kết hợp các trường thông tin một cách đầy đủ và chính xác nhất, ưu tiên thông tin chi tiết hơn nếu có sự khác biệt).

Dữ liệu đầu vào:

1. Nội dung bài báo cũ và custom info tương ứng

Nội dung bài báo:

{old_article_text}

personal_info:

{Utils.json_to_str(old_personal_info)}

2. Nội dung bài báo mới và personal_info tương ứng

Nội dung bài báo:

{new_article_text}

personal_info:

{Utils.json_to_str(new_personal_info)}


Kết quả mong muốn:

Một mảng JSON chứa các đối tượng, mỗi đối tượng có các trường: personal_id (từ dữ liệu mới), per_id (từ dữ liệu cũ) và các trường thông tin cá nhân được tổng hợp từ cả hai nguồn (full_name, birth_year_or_age, gender, occupation_or_position, organization, hometown_or_residence, personal_relationships).

Ví dụ:

[

{{

"personal_id": "2",

"per_id": "user_1752230968010",

"full_name": "Trương Mỹ Lan",

"birth_year_or_age": null,

"gender": "Female",

"occupation_or_position": "Chủ tịch Tập đoàn",

"organization": "Tập đoàn Vạn Thịnh Phát",

"hometown_or_residence": "TP Hồ Chí Minh",

"personal_relationships": null

}}

]

"""

In [24]:
print(prompt)


Mục tiêu:

Sử dụng cả hai thông tin bài báo và thông tin khách hàng để so sánh danh sách personal_info từ dữ liệu cũ và dữ liệu mới nhằm xác định các cá nhân trùng lặp. Đối với mỗi cá nhân trùng lặp được tìm thấy, hãy trả về một đối tượng JSON chứa personal_id của họ từ dữ liệu mới, per_unix_id của họ từ dữ liệu cũ, và một bản tổng hợp thông tin cá nhân của họ từ cả hai nguồn (kết hợp các trường thông tin một cách đầy đủ và chính xác nhất, ưu tiên thông tin chi tiết hơn nếu có sự khác biệt).

Dữ liệu đầu vào:

1. Nội dung bài báo cũ và custom info tương ứng

Nội dung bài báo:


Phó chủ nhiệm Ủy ban Pháp luật và Tư pháp của Quốc hội Nguyễn Phương Thủy giải đáp câu hỏi bà Trương Mỹ Lan có được chuyển từ án tử hình xuống chung thân. Trưa 27-6, tổng thư ký Quốc hội tổ chức họp báo công bố kết quả kỳ họp thứ 9, Quốc hội khóa XV. Tại cuộc họp báo, một phóng viên nước ngoài đặt vấn đề Luật sửa đổi, bổ sung một số điều của Bộ luật Hình sự vừa được Quốc hội thông qua tại kỳ họp đã bỏ án tử hìn

In [25]:
# Gọi mô hình
try:
    print("🚀 Đang gọi mô hình Bedrock ...")
    result, thinking = manager.generate_deepseek(prompt=prompt)
    print("\n✅ Kết quả phản hồi:")
    print(f"Thinking: {thinking}")
    print(f"Result: {result}")
except Exception as e:
    print(f"❌ Lỗi khi gọi mô hình: {e}")

🚀 Đang gọi mô hình Bedrock ...


<DEBUG-llm_model.bedrock_manager> - [DeepSeek] Đã nhận phản hồi: ```json
[
  {
    "personal_id": "P001",
    "per_id": "per_id_1752384078640389",
    "full_name": "Trương Mỹ Lan",
    "birth_year_or_age": null,
    "gender": "Female",
    "occupation_or_position": "Chủ tịch hội đồng quản trị",
    "organization": "Tập đoàn Vạn Thịnh Phát",
    "hometown_or_residence": null,
    "personal_relationships": "Bị cáo chính; Cổ đông lớn Ngân hàng SCB; Chỉ đạo các bị cáo Đinh Văn Thành, Bùi Anh Dũng, Võ Tấn Hoàng Văn, Tạ Chiêu Trung, Trương Khánh Hoàng, Trần Thị Mỹ Dung"
  }
]
```
<DEBUG-llm_model.bedrock_manager> - [DeepSeek] Reasoning: Okay, let's tackle this problem. So, the goal is to compare the personal_info from the old and new data to find duplicates. For each duplicate, I need to create a JSON object with the new personal_id, old per_id, and merged personal info, prioritizing more detailed info when there's a conflict.

First, I need to understand the data structures. The old persona


✅ Kết quả phản hồi:
Thinking: Okay, let's tackle this problem. So, the goal is to compare the personal_info from the old and new data to find duplicates. For each duplicate, I need to create a JSON object with the new personal_id, old per_id, and merged personal info, prioritizing more detailed info when there's a conflict.

First, I need to understand the data structures. The old personal_info has entries like Nguyễn Phương Thủy and Trương Mỹ Lan. The new personal_info has a lot more entries, but the main one that might overlap is Trương Mỹ Lan again. The other entries in the new data are probably new people, so they might not have duplicates in the old data.

So, the first step is to find which entries in the new data have matching entries in the old data. The key here is to match based on full_name, since that's the most consistent field. Let's check:

In the old data, Trương Mỹ Lan is present with per_id "per_id_1752384078640389". In the new data, she's P001. So that's a clear mat

### Extract resp

In [26]:
new_personal_info

[{'personal_id': 'P001',
  'full_name': 'Trương Mỹ Lan',
  'birth_year_or_age': None,
  'gender': 'Female',
  'occupation_or_position': 'Chủ tịch hội đồng quản trị',
  'organization': 'Tập đoàn Vạn Thịnh Phát',
  'hometown_or_residence': None,
  'personal_relationships': 'Bị cáo chính; Cổ đông lớn Ngân hàng SCB; Chỉ đạo các bị cáo Đinh Văn Thành, Bùi Anh Dũng, Võ Tấn Hoàng Văn, Tạ Chiêu Trung, Trương Khánh Hoàng, Trần Thị Mỹ Dung'},
 {'personal_id': 'P002',
  'full_name': 'Đinh Văn Thành',
  'birth_year_or_age': None,
  'gender': 'Male',
  'occupation_or_position': None,
  'organization': None,
  'hometown_or_residence': None,
  'personal_relationships': 'Đồng phạm của Trương Mỹ Lan'},
 {'personal_id': 'P003',
  'full_name': 'Bùi Anh Dũng',
  'birth_year_or_age': None,
  'gender': 'Male',
  'occupation_or_position': None,
  'organization': None,
  'hometown_or_residence': None,
  'personal_relationships': 'Đồng phạm của Trương Mỹ Lan'},
 {'personal_id': 'P004',
  'full_name': 'Võ Tấn H

In [27]:
print(result)

```json
[
  {
    "personal_id": "P001",
    "per_id": "per_id_1752384078640389",
    "full_name": "Trương Mỹ Lan",
    "birth_year_or_age": null,
    "gender": "Female",
    "occupation_or_position": "Chủ tịch hội đồng quản trị",
    "organization": "Tập đoàn Vạn Thịnh Phát",
    "hometown_or_residence": null,
    "personal_relationships": "Bị cáo chính; Cổ đông lớn Ngân hàng SCB; Chỉ đạo các bị cáo Đinh Văn Thành, Bùi Anh Dũng, Võ Tấn Hoàng Văn, Tạ Chiêu Trung, Trương Khánh Hoàng, Trần Thị Mỹ Dung"
  }
]
```


In [28]:
matched_item = Utils.json_str_to_dict(result)